# Ark Export Deep Dive

This notebook provides a deep dive into the Ark export pipeline, CDG construction, and validation for analog circuit compilation.

**Goal:** Understand how to export trained ODE-based models to real analog circuits using the Ark framework.

**This notebook is primarily for hardware architects** implementing circuit primitives for ODE-based neural computations. Neural architects may use this to understand how their ODE-based models are compiled to analog circuits.

## What is Ark?

Ark is an analog circuit compiler for ODEs (Arco/Legno framework). It:

- **Compiles ODEs to analog circuits:** dx/dt = f_θ(x,t) → BaseAnalogCkt subclass
- **Generates JAX code:** Produces executable circuit specifications
- **Supports mismatch injection:** Stochastic forward passes for validation
- **Validates against digital:** Compare analog vs digital behavior

**Ark is the compiler that bridges neural architects' ODE models and hardware architects' circuit primitives.**

**When to use Ark:** When you have a trained ODE-based model and want to compile it to a real analog circuit.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import numpy as np

# Note: Ark export requires Ark installation (JAX, diffrax)
# This notebook shows the API; actual execution requires Ark dependencies
# from neuro_analog.ark_bridge.neural_ode_cdg import export_neural_ode_to_ark

## Prerequisites

To use Ark export, you need:

- **JAX:** For automatic differentiation and JIT compilation
- **diffrax:** For ODE solvers in JAX
- **Ark:** The analog circuit compiler (install from Ark repository)

```bash
pip install jax jaxlib diffrax
# Install Ark from https://github.com/ark-circuit-compiler/ark
```

## CDG (Constrained Dynamical Graph) Construction

The Constrained Dynamical Graph maps a neural ODE to circuit topology:

- **Nodes:** ODE function evaluations (f_θ)
- **Edges:** State transitions (x_t → x_{t+dt})
- **Constraints:** Circuit capacity limits (crossbar size, integrator count)
- **Parameters:** Weight matrices mapped to crossbar conductances

In [ ]:
# Example: CDG construction (pseudo-code)
# from neuro_analog.ark_bridge.neural_ode_cdg import build_cdg

# cdg = build_cdg(
#     neural_ode_model,
#     state_dim=64,
#     time_span=(0.0, 1.0),
#     crossbar_rows=256,
#     crossbar_cols=256,
# )

# print(f"CDG has {len(cdg.nodes)} nodes")
# print(f"CDG has {len(cdg.edges)} edges")
# print(f"State dimension: {cdg.state_dim}")

print("CDG construction requires Ark installation.")
print("See examples/03_ark_pipeline.py for a working example.")

## export_neural_ode_to_ark() Walkthrough

The main export function takes a Neural ODE model and generates a BaseAnalogCkt subclass:

In [ ]:
# Example: Neural ODE export (pseudo-code)
# from neuro_analog.ark_bridge.neural_ode_cdg import export_neural_ode_to_ark

# # Assume we have a trained Neural ODE model
# neural_ode_model = ...  # Your trained model

# # Export to Ark
# ark_circuit = export_neural_ode_to_ark(
#     neural_ode_model,
#     state_dim=64,
#     time_span=(0.0, 1.0),
#     dt=0.01,
# )

# print(f"Generated BaseAnalogCkt subclass: {ark_circuit.__name__}")
# print(f"Circuit has {ark_circuit.num_integrators} integrators")
# print(f"Circuit has {ark_circuit.num_crossbars} crossbars")

print("Neural ODE export requires Ark installation.")
print("See examples/03_ark_pipeline.py for a working example.")

## Generated BaseAnalogCkt Code Structure

The export generates a JAX-based circuit specification:

In [ ]:
# Example generated code structure (pseudo-code)

generated_code = """
class NeuralODEAnalogCkt(BaseAnalogCkt):
    def __init__(self, params):
        self.params = params
        self.integrators = [Integrator() for _ in range(64)]
        self.crossbars = [Crossbar(params[f'W{i}']) for i in range(3)]
    
    def forward(self, x0, t_span):
        return self._solve_ode(x0, t_span, sigma=0.0)
    
    def forward_with_mismatch(self, x0, t_span, sigma=0.05):
        return self._solve_ode(x0, t_span, sigma=sigma)
    
    def _solve_ode(self, x0, t_span, sigma):
        pass
"""

print("Generated BaseAnalogCkt structure:")
print(generated_code)

## Mismatch Injection in Ark

Ark supports stochastic forward passes to simulate analog nonidealities:

In [ ]:
# Example: Mismatch injection (pseudo-code)

# # Nominal forward pass (no mismatch)
# nominal_out = ark_circuit.forward(x0, t_span=(0.0, 1.0))

# # Stochastic forward pass with 5% mismatch
# mismatch_out = ark_circuit.forward_with_mismatch(x0, t_span=(0.0, 1.0), sigma=0.05)

# # Compare outputs
# error = jnp.linalg.norm(nominal_out - mismatch_out)
# print(f"Mismatch-induced error: {error:.4f}")

print("Mismatch injection requires Ark installation.")
print("See examples/03_ark_pipeline.py for a working example.")

## Running Exported Circuits Against Ark

Once exported, you can validate the circuit against Ark:

In [ ]:
# Example: Validation (pseudo-code)

# # Run multiple stochastic trials
# n_trials = 50
# outputs = []
# for _ in range(n_trials):
#     out = ark_circuit.forward_with_mismatch(x0, t_span=(0.0, 1.0), sigma=0.05)
#     outputs.append(out)

# # Compute statistics
# outputs = jnp.stack(outputs)
# mean_out = jnp.mean(outputs, axis=0)
# std_out = jnp.std(outputs, axis=0)

# print(f"Mean output: {mean_out}")
# print(f"Std output: {std_out}")

print("Validation requires Ark installation.")
print("See examples/03_ark_pipeline.py for a working example.")

## Ark Compatibility by Architecture

Not all architectures have full Ark export support:

In [ ]:
ark_compatibility = [
    ("Neural ODE", "Perfect fit", "dx/dt = f_θ(x,t) IS Ark input format"),
    ("SSM", "Strong fit", "Export via S4D to CDG"),
    ("DEQ", "Strong fit", "Export via gradient-flow ODE to CDG"),
    ("EBM", "Strong fit", "Export via Hopfield to CDG bridge"),
    ("Diffusion", "Strong fit", "Export via VP-SDE prob-flow to CDG"),
    ("Flow", "Strong fit", "Export via neural_ode (velocity is MLP)"),
    ("Transformer", "Analysis-only", "No ODE form; FFN partition is analysis-only"),
]

print("Ark compatibility by architecture:")
for arch, level, reason in ark_compatibility:
    print(f"  {arch:20s} ({level:15s}): {reason}")

## Limitations and Prerequisites

**Limitations:**
- **CDG capacity constraint:** Large models exceed crossbar capacity
- **Experiment models only:** Current export works for small MLPs
- **Architecture-specific:** Not all architectures have full export support

**Prerequisites:**
- Ark installation (from Ark repository)
- JAX and diffrax
- Trained ODE-based model
- Sufficient compute for JAX JIT compilation

## Links to Other Architecture Examples

For Ark export workflows for other architectures, see:

- **Neural ODE:** `examples/03_ark_pipeline.py` (full pipeline)
- **SSM:** `examples/11_ssm_ark.py` (S4D export)
- **DEQ:** `examples/10_deq_ark.py` (gradient-flow ODE export)
- **EBM:** `examples/06_ebm_ark.py` (Hopfield CDG bridge)
- **Diffusion:** `examples/08_diffusion_ark.py` (VP-SDE export)
- **Flow:** `examples/07_flow_ark.py` (neural_ode export)
- **Transformer FFN:** `examples/09_transformer_ffn_ark.py` (FFN partition)
- **CDG bridge:** `examples/05_cdg_bridge.py` (low-level CDG construction)

## Full Pipeline Example

The complete Ark export pipeline:

1. **Train model:** Train your Neural ODE/SSM/DEQ/etc. in PyTorch
2. **Extract IR:** Use architecture-specific extractor to build AnalogGraph
3. **Build CDG:** Construct Constrained Dynamical Graph from IR
4. **Export to Ark:** Generate BaseAnalogCkt subclass
5. **Validate:** Run nominal and mismatch forward passes
6. **Compare:** Compare analog vs digital behavior

See `examples/03_ark_pipeline.py` for a complete working example.

## Key Takeaways

1. **Ark compiles ODEs to analog circuits** - dx/dt = f_θ(x,t) to BaseAnalogCkt
2. **CDG maps neural ODE to circuit topology** - nodes, edges, constraints, parameters
3. **export_neural_ode_to_ark() generates JAX code** - executable circuit specification
4. **Mismatch injection enables validation** - stochastic forward passes simulate analog noise
5. **Ark compatibility varies by architecture** - Neural ODE (perfect) to Transformer (analysis-only)
6. **Limitations exist** - CDG capacity constraint, experiment models only
7. **Prerequisites required** - Ark, JAX, diffrax installation
8. **Full pipeline in examples/03_ark_pipeline.py** - complete working example